In [1]:
import torch

if torch.backends.mps.is_available():
    print("✅ MPS (Apple GPU) is AVAILABLE")
else:
    print("❌ MPS NOT available")

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

✅ MPS (Apple GPU) is AVAILABLE
Using device: mps


In [4]:
"""
Stage 1 — EfficientNet-B3 Multi-Label Training (Upgraded)
===========================================================
Improvements over original:
  1. True multi-label inference via sigmoid + per-class threshold tuning
  2. Full multi-label metrics: micro/macro F1, mAP, Hamming loss, exact match
  3. Validation on BOTH single-label accuracy AND multi-label mAP
  4. 50 epochs with warmup + cosine annealing + early stopping (patience=10)
  5. EfficientNet-B3 (stronger than B0, better feature extraction at 300px)
  6. Plant-disease-optimized augmentation (RandAugment + RandomErasing)
  7. Asymmetric Loss (ASL) for multi-label class imbalance
  8. Per-class threshold tuning on validation set after training
  9. Label co-occurrence via correlation-aware pos_weight
 10. MixUp + CutMix at batch level

Target: ~97% single-label accuracy, mAP > 0.90

Usage:
  python stage1_train_v2.py

Requirements:
  pip install torch torchvision scikit-learn pandas tqdm timm
"""

import os, json, time, csv, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
from contextlib import nullcontext
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    f1_score, average_precision_score, hamming_loss,
)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, ConcatDataset
from torchvision import datasets, transforms, models

warnings.filterwarnings("ignore")

# ══════════════════════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════════════════════

DATASET_DIR   = "dataset_balanced"
CSV_PATH      = "dataset_balanced/multilabel.csv"
MODEL_DIR     = "models_v2"
MODEL_PATH    = f"{MODEL_DIR}/best_model.pth"
CLASS_MAP     = f"{MODEL_DIR}/class_map.json"
LOG_PATH      = f"{MODEL_DIR}/training_log.json"
THRESH_PATH   = f"{MODEL_DIR}/thresholds.json"
PLOTS_DIR     = f"{MODEL_DIR}/plots"

IMG_SIZE      = 300        # EfficientNet-B3 native resolution
BATCH_SIZE    = 24         # smaller batch due to larger model + resolution
EPOCHS        = 20
WARMUP_EPOCHS = 5
LR_HEAD       = 3e-4
LR_BACKBONE   = 5e-5
WEIGHT_DECAY  = 1e-4
PATIENCE      = 10         # early stopping patience
ALPHA         = 0.7        # CE weight vs BCE weight
DEFAULT_THRESH = 0.5
NUM_WORKERS   = 0          # set to 0 for MPS (Mac), 4 for CUDA

ALL_CLASSES = [
    "Anthracnose", "BacterialBlight", "BlackSpot", "Canker",
    "CurlVirus", "DeficiencyLeaf", "DryLeaf", "Greening",
    "Healthy", "LeafMiner", "Melanose", "SootyMould", "SpiderMites"
]
NUM_CLASSES   = len(ALL_CLASSES)
CLASS_TO_IDX  = {c: i for i, c in enumerate(ALL_CLASSES)}

# Dataset-specific normalisation stats
try:
    from normalization_stats import MEAN, STD
    print("✓ Using dataset-specific normalisation stats")
except ImportError:
    MEAN = [0.6753, 0.7039, 0.5966]
    STD  = [0.3517, 0.3215, 0.4195]
    print("  Using citrus-specific defaults")


# ══════════════════════════════════════════════════════════════════════════════
# DEVICE
# ══════════════════════════════════════════════════════════════════════════════

def get_device():
    if torch.backends.mps.is_available(): return torch.device("mps")
    if torch.cuda.is_available():         return torch.device("cuda")
    return torch.device("cpu")

DEVICE = get_device()

def autocast_ctx():
    if DEVICE.type == "cuda":
        return torch.autocast(device_type="cuda", dtype=torch.float16)
    return nullcontext()

print("CSV_PATH:", CSV_PATH)
print("DATASET_DIR:", DATASET_DIR)
# ══════════════════════════════════════════════════════════════════════════════
# AUGMENTATION — plant-disease optimised
# ══════════════════════════════════════════════════════════════════════════════

def get_transforms(split: str) -> transforms.Compose:
    normalize = transforms.Normalize(mean=MEAN, std=STD)

    if split == "train":
        return transforms.Compose([
            transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
            transforms.RandomCrop(IMG_SIZE),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.5),
            transforms.RandomRotation(degrees=30),
            # Colour augmentation — important for disease colour changes
            transforms.ColorJitter(
                brightness=0.35,
                contrast=0.35,
                saturation=0.30,
                hue=0.12,
            ),
            # RandAugment — learns best augmentation policy automatically
            transforms.RandAugment(num_ops=2, magnitude=9),
            # Random erasing simulates partial occlusion of lesion areas
            transforms.ToTensor(),
            normalize,
            transforms.RandomErasing(p=0.25, scale=(0.02, 0.15)),
        ])
    else:
        return transforms.Compose([
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.ToTensor(),
            normalize,
        ])


# ══════════════════════════════════════════════════════════════════════════════
# MIXUP + CUTMIX (batch-level runtime augmentation)
# ══════════════════════════════════════════════════════════════════════════════

def mixup(imgs, labels, alpha=0.4):
    """Works for both hard labels (long) and soft labels (float tensor)."""
    lam   = np.random.beta(alpha, alpha)
    idx   = torch.randperm(imgs.size(0), device=imgs.device)
    mixed = lam * imgs + (1 - lam) * imgs[idx]
    if labels.dtype == torch.long:
        # Convert to one-hot for mixing
        oh = F.one_hot(labels, NUM_CLASSES).float()
        mixed_labels = lam * oh + (1 - lam) * oh[idx]
    else:
        mixed_labels = lam * labels + (1 - lam) * labels[idx]
    return mixed, mixed_labels, True   # True = soft labels


def cutmix(imgs, labels, alpha=1.0):
    lam   = np.random.beta(alpha, alpha)
    idx   = torch.randperm(imgs.size(0), device=imgs.device)
    _, _, H, W = imgs.shape
    cut_h = int(H * np.sqrt(1 - lam))
    cut_w = int(W * np.sqrt(1 - lam))
    cx    = np.random.randint(W)
    cy    = np.random.randint(H)
    x1, x2 = np.clip(cx - cut_w//2, 0, W), np.clip(cx + cut_w//2, 0, W)
    y1, y2 = np.clip(cy - cut_h//2, 0, H), np.clip(cy + cut_h//2, 0, H)
    mixed = imgs.clone()
    mixed[:, :, y1:y2, x1:x2] = imgs[idx, :, y1:y2, x1:x2]
    lam_actual = 1 - (x2 - x1) * (y2 - y1) / (W * H)
    if labels.dtype == torch.long:
        oh = F.one_hot(labels, NUM_CLASSES).float()
        mixed_labels = lam_actual * oh + (1 - lam_actual) * oh[idx]
    else:
        mixed_labels = lam_actual * labels + (1 - lam_actual) * labels[idx]
    return mixed, mixed_labels, True


# ══════════════════════════════════════════════════════════════════════════════
# ASYMMETRIC LOSS (ASL) — handles multi-label class imbalance better than BCE
# ASL down-weights easy negatives and focuses on hard positives
# Reference: Ridnik et al., 2021 (ASL paper)
# ══════════════════════════════════════════════════════════════════════════════

class AsymmetricLoss(nn.Module):
    def __init__(self, gamma_neg=4, gamma_pos=1, clip=0.05, eps=1e-8):
        """
        gamma_neg: focusing param for negatives (down-weight easy negatives)
        gamma_pos: focusing param for positives (standard focal)
        clip:      probability margin to shift negatives away from decision boundary
        eps:       numerical stability epsilon
        """
        super().__init__()
        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip      = clip
        self.eps       = eps

    def forward(self, logits, targets):
        logits  = torch.nan_to_num(logits.float(), nan=0.0, posinf=20.0, neginf=-20.0)
        logits  = logits.clamp(min=-20.0, max=20.0)
        targets = torch.nan_to_num(targets.float(), nan=0.0, posinf=1.0, neginf=0.0)
        targets = targets.clamp_(0.0, 1.0)

        xs_pos = torch.sigmoid(logits).clamp(min=self.eps, max=1.0 - self.eps)
        xs_neg = (1.0 - xs_pos).clamp(min=self.eps, max=1.0)
        if self.clip is not None and self.clip > 0:
            xs_neg = (xs_neg + self.clip).clamp(max=1.0)

        loss_pos = targets * torch.log(xs_pos)
        loss_neg = (1.0 - targets) * torch.log(xs_neg.clamp(min=self.eps))
        loss = loss_pos + loss_neg

        pt = xs_pos * targets + xs_neg * (1.0 - targets)
        gamma = self.gamma_pos * targets + self.gamma_neg * (1.0 - targets)
        one_sided_w = torch.pow((1.0 - pt).clamp(min=0.0, max=1.0), gamma)
        loss = -loss * one_sided_w

        return torch.nan_to_num(loss, nan=0.0, posinf=0.0, neginf=0.0).mean()


# ══════════════════════════════════════════════════════════════════════════════
# DATASETS
# ══════════════════════════════════════════════════════════════════════════════

class MultiLabelDataset(Dataset):
    """
    Loads all images from multilabel.csv for a given split.
    Handles both single-label class folders and multi-label images/ folder.
    """
    def __init__(self, split: str, transform=None):
        self.transform = transform or get_transforms(split)
        self.samples   = []

        if not Path(CSV_PATH).exists():
            print(f"  [WARN] {CSV_PATH} not found")
            return

        df       = pd.read_csv(CSV_PATH)
        split_df = df[df["split"] == split].reset_index(drop=True)

        for _, row in split_df.iterrows():
            img_path = Path(DATASET_DIR) / row["image_path"]
            if not img_path.exists():
                continue
            label_vec = torch.tensor(
                [float(row[cls]) for cls in ALL_CLASSES],
                dtype=torch.float32
            )
            self.samples.append((img_path, label_vec))

        multi = sum(1 for _, lv in self.samples if lv.sum() > 1)
        print(f"  {split:5s}: {len(self.samples):6d} images  "
              f"({multi} multi-label, {len(self.samples)-multi} single-label)")

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, label_vec = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label_vec


def make_pos_weight(csv_path: str, split: str = "train") -> torch.Tensor:
    """
    Compute pos_weight per class accounting for label co-occurrences.
    pos_weight[c] = neg_count[c] / pos_count[c]
    """
    df  = pd.read_csv(csv_path)
    df  = df[df["split"] == split]
    pos = df[ALL_CLASSES].sum(axis=0).values.astype(float)
    neg = len(df) - pos
    pw  = neg / np.maximum(pos, 1)
    return torch.tensor(pw, dtype=torch.float32)


def make_class_weights(csv_path: str, split: str = "train") -> torch.Tensor:
    """
    For single-label CrossEntropyLoss — inverse frequency weighting.
    """
    df  = pd.read_csv(csv_path)
    df  = df[(df["split"] == split) & (df[ALL_CLASSES].sum(axis=1) == 1)]
    counts = np.array([
        len(df[df[cls] == 1]) for cls in ALL_CLASSES
    ], dtype=float)
    counts = np.maximum(counts, 1)
    weights = 1.0 / counts
    weights /= weights.sum()
    return torch.tensor(weights * NUM_CLASSES, dtype=torch.float32)


# ══════════════════════════════════════════════════════════════════════════════
# MODEL — EfficientNet-B3
# ══════════════════════════════════════════════════════════════════════════════

def build_model() -> nn.Module:
    """
    EfficientNet-B3:
      - Native 300×300 input (vs B0's 224×224)
      - 12M params vs 5.3M (better feature capacity)
      - Top-1 ImageNet accuracy: 82.1% vs 77.7%
      - Single head: 13 logits used for both CE and BCE loss
    """
    model = models.efficientnet_b3(
        weights=models.EfficientNet_B3_Weights.IMAGENET1K_V1
    )
    # Freeze backbone initially for warmup
    for p in model.features.parameters():
        p.requires_grad = False

    in_feats = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.4),
        nn.Linear(in_feats, 512),
        nn.SiLU(),                     # smooth activation (used in EfficientNet)
        nn.Dropout(p=0.2),
        nn.Linear(512, NUM_CLASSES),
        # No sigmoid — applied externally during inference/loss
    )
    return model.to(DEVICE)


# ══════════════════════════════════════════════════════════════════════════════
# METRICS
# ══════════════════════════════════════════════════════════════════════════════

def compute_multilabel_metrics(labels: np.ndarray,
                                probs: np.ndarray,
                                thresholds=None) -> dict:
    """
    labels : (N, 13) binary ground truth
    probs  : (N, 13) sigmoid probabilities
    thresholds: (13,) per-class thresholds, defaults to 0.5

    Returns dict of all multi-label metrics.
    """
    if thresholds is None:
        thresholds = np.full(NUM_CLASSES, DEFAULT_THRESH)

    labels = (labels > 0.5).astype(int)
    probs = np.asarray(probs, dtype=np.float32)
    probs = np.nan_to_num(probs, nan=0.0, posinf=1.0, neginf=0.0)
    probs = np.clip(probs, 0.0, 1.0)
    thresholds = np.asarray(thresholds, dtype=np.float32)
    thresholds = np.nan_to_num(thresholds, nan=DEFAULT_THRESH,
                               posinf=1.0, neginf=0.0)
    thresholds = np.clip(thresholds, 0.0, 1.0)

    preds = (probs >= thresholds).astype(int)

    # Mean Average Precision
    per_ap = []
    for c in range(NUM_CLASSES):
        if labels[:, c].sum() > 0:
            per_ap.append(average_precision_score(labels[:, c], probs[:, c]))
    mAP = float(np.mean(per_ap)) if per_ap else 0.0

    return {
        "mAP"         : mAP,
        "macro_f1"    : f1_score(labels, preds, average="macro",  zero_division=0),
        "micro_f1"    : f1_score(labels, preds, average="micro",  zero_division=0),
        "hamming_loss": hamming_loss(labels, preds),
        "exact_match" : float((preds == labels).all(axis=1).mean()),
        "per_class_ap": {
            ALL_CLASSES[c]: round(float(average_precision_score(
                labels[:, c], probs[:, c])), 4)
            for c in range(NUM_CLASSES) if labels[:, c].sum() > 0
        }
    }


def tune_thresholds(labels: np.ndarray, probs: np.ndarray) -> np.ndarray:
    """
    Per-class threshold tuning: find threshold that maximises F1 per class.
    Run on validation set after training.
    """
    labels = (labels > 0.5).astype(int)
    probs = np.asarray(probs, dtype=np.float32)
    probs = np.nan_to_num(probs, nan=0.0, posinf=1.0, neginf=0.0)
    probs = np.clip(probs, 0.0, 1.0)

    best_thresholds = np.full(NUM_CLASSES, DEFAULT_THRESH)
    candidates      = np.arange(0.1, 0.95, 0.05)

    for c in range(NUM_CLASSES):
        if labels[:, c].sum() == 0:
            continue
        best_f1 = 0
        for t in candidates:
            preds = (probs[:, c] >= t).astype(int)
            f1    = f1_score(labels[:, c], preds, zero_division=0)
            if f1 > best_f1:
                best_f1              = f1
                best_thresholds[c]   = t

    print("\n  Per-class optimal thresholds:")
    for c, (cls, t) in enumerate(zip(ALL_CLASSES, best_thresholds)):
        print(f"    {cls:<22} t={t:.2f}")
    return best_thresholds


# ══════════════════════════════════════════════════════════════════════════════
# TRAIN LOOP
# ══════════════════════════════════════════════════════════════════════════════

def train_one_epoch(model, loader, ce_criterion, asl_criterion,
                    optimizer, use_mix=True) -> dict:
    model.train()
    total_loss = sl_correct = sl_total = ml_total = 0
    skipped_batches = 0

    for imgs, label_vecs in loader:
        imgs       = imgs.to(DEVICE, non_blocking=True)
        label_vecs = label_vecs.to(DEVICE, non_blocking=True)

        # Identify single vs multi-label in this batch
        is_single = label_vecs.sum(dim=1) == 1

        # Apply MixUp or CutMix randomly (50% each, 60% of batches total)
        soft = False
        mixed_labels = label_vecs
        if use_mix and np.random.rand() < 0.6:
            if np.random.rand() < 0.5:
                imgs, mixed_labels, soft = mixup(imgs, label_vecs)
            else:
                imgs, mixed_labels, soft = cutmix(imgs, label_vecs)

        optimizer.zero_grad()
        with autocast_ctx():
            logits = model(imgs)
            logits = torch.nan_to_num(logits, nan=0.0, posinf=20.0, neginf=-20.0)

            # ── Single-label loss (CrossEntropy) ──────────────────────────
            if is_single.any() and not soft:
                sl_logits = logits[is_single]
                # Convert one-hot back to class indices for CE
                sl_labels = label_vecs[is_single].argmax(dim=1)
                ce_loss   = ce_criterion(sl_logits, sl_labels) * ALPHA
            else:
                ce_loss = torch.tensor(0.0, device=DEVICE)

            # ── Multi-label loss (ASL) — all images ───────────────────────
            if soft:
                asl_loss = asl_criterion(logits, mixed_labels) * (1 - ALPHA)
            else:
                asl_loss = asl_criterion(logits, label_vecs) * (1 - ALPHA)

            loss = ce_loss + asl_loss

        if not torch.isfinite(loss):
            skipped_batches += 1
            optimizer.zero_grad(set_to_none=True)
            continue

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item() * imgs.size(0)
        if is_single.any() and not soft:
            sl_logits = logits[is_single].detach()
            sl_labels = label_vecs[is_single].argmax(dim=1)
            sl_correct += (sl_logits.argmax(1) == sl_labels).sum().item()
            sl_total   += is_single.sum().item()
        ml_total += (~is_single).sum().item()

    n = sl_total + ml_total
    return {
        "loss"    : total_loss / max(n, 1),
        "sl_acc"  : sl_correct / max(sl_total, 1),
        "sl_total": sl_total,
        "ml_total": ml_total,
        "skipped_batches": skipped_batches,
    }


@torch.no_grad()
def evaluate(model, loader) -> tuple[dict, np.ndarray, np.ndarray]:
    model.eval()
    all_labels = []
    all_probs  = []
    sanitized_batches = 0

    for imgs, label_vecs in loader:
        imgs       = imgs.to(DEVICE, non_blocking=True)
        label_vecs = label_vecs.to(DEVICE, non_blocking=True)
        with autocast_ctx():
            logits = model(imgs)
        if not torch.isfinite(logits).all():
            sanitized_batches += 1
            logits = torch.nan_to_num(logits, nan=0.0, posinf=20.0, neginf=-20.0)
        probs = torch.sigmoid(logits).cpu().float().numpy()
        probs = np.nan_to_num(probs, nan=0.0, posinf=1.0, neginf=0.0)
        probs = np.clip(probs, 0.0, 1.0)
        all_probs.append(probs)
        all_labels.append(label_vecs.cpu().numpy())

    probs = np.concatenate(all_probs, axis=0)
    labels = np.concatenate(all_labels, axis=0)
    labels = (labels > 0.5).astype(int)

    metrics = compute_multilabel_metrics(labels, probs)
    metrics["sanitized_batches"] = sanitized_batches

    # Single-label accuracy (for images with exactly one label)
    is_single = labels.sum(axis=1) == 1
    if is_single.any():
        sl_true  = labels[is_single].argmax(axis=1)
        sl_pred  = probs[is_single].argmax(axis=1)
        sl_acc   = (sl_true == sl_pred).mean()
        metrics["sl_acc"] = float(sl_acc)
    else:
        metrics["sl_acc"] = 0.0

    return metrics, labels, probs


# ══════════════════════════════════════════════════════════════════════════════
# MAIN TRAINING FUNCTION
# ══════════════════════════════════════════════════════════════════════════════

def train():
    os.makedirs(MODEL_DIR, exist_ok=True)
    os.makedirs(PLOTS_DIR, exist_ok=True)
    print(f"\n  Device : {DEVICE}")
    print(f"  Model  : EfficientNet-B3  |  Input: {IMG_SIZE}×{IMG_SIZE}")
    print(f"  Epochs : {EPOCHS}  |  Batch: {BATCH_SIZE}  |  LR_head: {LR_HEAD}")

    # ── Datasets ─────────────────────────────────────────────────────────────
    print("\nLoading datasets from multilabel.csv ...")
    train_ds  = MultiLabelDataset("train", get_transforms("train"))
    val_ds    = MultiLabelDataset("val",   get_transforms("val"))

    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"),
        drop_last=False,
    )
    val_loader = DataLoader(
        val_ds, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"),
    )

    # ── Model ─────────────────────────────────────────────────────────────────
    model = build_model()
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\n  Params total={total:,}  trainable={trainable:,} (head only — warmup)")

    with open(CLASS_MAP, "w") as f:
        json.dump({
            "class_to_idx": CLASS_TO_IDX,
            "idx_to_class": {str(v): k for k, v in CLASS_TO_IDX.items()}
        }, f, indent=2)

    # ── Loss functions ─────────────────────────────────────────────────────────
    class_weights = make_class_weights(CSV_PATH).to(DEVICE)
    ce_criterion  = nn.CrossEntropyLoss(
        weight=class_weights, label_smoothing=0.1
    )
    asl_criterion = AsymmetricLoss(gamma_neg=4, gamma_pos=1, clip=0.05)

    # ── Optimiser — Phase 1: head only ────────────────────────────────────────
    head_optimizer = torch.optim.AdamW(
        model.classifier.parameters(), lr=LR_HEAD, weight_decay=WEIGHT_DECAY
    )
    warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
        head_optimizer, start_factor=0.1, end_factor=1.0,
        total_iters=WARMUP_EPOCHS
    )

    best_mAP         = 0.0
    best_sl_acc      = 0.0
    patience_counter = 0
    log              = []

    print(f"\n{'═'*72}")
    print(f"  PHASE 1 — Warmup ({WARMUP_EPOCHS} epochs, head only, backbone frozen)")
    print(f"{'═'*72}")
    print(f"  {'Ep':>3}  {'Loss':>8}  {'SL_Acc':>8}  {'Val_mAP':>9}  "
          f"{'Val_SL':>8}  {'Val_F1':>8}")

    for epoch in range(1, WARMUP_EPOCHS + 1):
        t0  = time.time()
        tr  = train_one_epoch(model, train_loader, ce_criterion,
                               asl_criterion, head_optimizer, use_mix=False)
        vm, _, _ = evaluate(model, val_loader)
        warmup_scheduler.step()

        print(f"  {epoch:>3}  {tr['loss']:>8.4f}  {tr['sl_acc']:>8.4f}  "
              f"{vm['mAP']:>9.4f}  {vm['sl_acc']:>8.4f}  "
              f"{vm['macro_f1']:>8.4f}  [{time.time()-t0:.0f}s]")
        if tr['skipped_batches'] or vm['sanitized_batches']:
            print(f"       warning: skipped_train_batches={tr['skipped_batches']}  "
                  f"sanitized_val_batches={vm['sanitized_batches']}")

    # ── Phase 2: full fine-tune ────────────────────────────────────────────────
    for p in model.parameters():
        p.requires_grad = True

    optimizer = torch.optim.AdamW([
        {"params": model.features.parameters(), "lr": LR_BACKBONE},
        {"params": model.classifier.parameters(), "lr": LR_HEAD},
    ], weight_decay=WEIGHT_DECAY)

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=EPOCHS - WARMUP_EPOCHS, eta_min=1e-6
    )

    remaining = EPOCHS - WARMUP_EPOCHS
    print(f"\n{'═'*72}")
    print(f"  PHASE 2 — Full fine-tune ({remaining} epochs, all layers)")
    print(f"  Backbone lr={LR_BACKBONE}  Head lr={LR_HEAD}")
    print(f"  Early stopping patience={PATIENCE} on val mAP")
    print(f"{'═'*72}")
    print(f"  {'Ep':>3}  {'Loss':>8}  {'SL_Acc':>8}  {'Val_mAP':>9}  "
          f"{'Val_SL':>8}  {'Val_F1':>8}")

    for epoch in range(1, remaining + 1):
        t0  = time.time()
        tr  = train_one_epoch(model, train_loader, ce_criterion,
                               asl_criterion, optimizer, use_mix=True)
        vm, val_labels, val_probs = evaluate(model, val_loader)
        scheduler.step()

        ep_total = epoch + WARMUP_EPOCHS
        marker   = ""

        # Save if improved on EITHER mAP or sl_acc
        improved = (vm["mAP"] > best_mAP or
                    (vm["mAP"] >= best_mAP - 0.002 and vm["sl_acc"] > best_sl_acc))
        if improved:
            best_mAP     = max(vm["mAP"], best_mAP)
            best_sl_acc  = max(vm["sl_acc"], best_sl_acc)
            torch.save({
                "epoch"      : ep_total,
                "model_state": model.state_dict(),
                "val_mAP"    : vm["mAP"],
                "val_sl_acc" : vm["sl_acc"],
                "val_macro_f1": vm["macro_f1"],
                "classes"    : ALL_CLASSES,
                "img_size"   : IMG_SIZE,
            }, MODEL_PATH)
            marker           = "  ✓ saved"
            patience_counter = 0
        else:
            patience_counter += 1

        print(f"  {ep_total:>3}  {tr['loss']:>8.4f}  {tr['sl_acc']:>8.4f}  "
              f"{vm['mAP']:>9.4f}  {vm['sl_acc']:>8.4f}  "
              f"{vm['macro_f1']:>8.4f}  [{time.time()-t0:.0f}s]{marker}")
        if tr['skipped_batches'] or vm['sanitized_batches']:
            print(f"       warning: skipped_train_batches={tr['skipped_batches']}  "
                  f"sanitized_val_batches={vm['sanitized_batches']}")

        log.append({
            "epoch"     : ep_total,
            "train_loss": round(tr["loss"],    5),
            "train_acc" : round(tr["sl_acc"],  5),
            "val_mAP"   : round(vm["mAP"],     5),
            "val_sl_acc": round(vm["sl_acc"],  5),
            "val_macro_f1": round(vm["macro_f1"], 5),
        })

        if patience_counter >= PATIENCE:
            print(f"\n  Early stopping at epoch {ep_total} (patience={PATIENCE})")
            break

    with open(LOG_PATH, "w") as f:
        json.dump(log, f, indent=2)

    print(f"\n  Best val mAP   : {best_mAP:.4f}")
    print(f"  Best val SL acc: {best_sl_acc:.4f}")
    print(f"  Model saved    : {MODEL_PATH}")
    return model


# ══════════════════════════════════════════════════════════════════════════════
# THRESHOLD TUNING + FINAL TEST EVALUATION
# ══════════════════════════════════════════════════════════════════════════════

def final_evaluate():
    print(f"\n{'═'*72}")
    print(f"  FINAL EVALUATION")
    print(f"{'═'*72}")

    ckpt  = torch.load(MODEL_PATH, map_location=DEVICE)
    model = build_model()
    model.load_state_dict(ckpt["model_state"])
    model.eval()
    print(f"  Loaded epoch {ckpt['epoch']} | val_mAP={ckpt['val_mAP']:.4f}")

    val_ds    = MultiLabelDataset("val",  get_transforms("val"))
    test_ds   = MultiLabelDataset("test", get_transforms("val"))
    val_loader  = DataLoader(val_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    # ── Tune thresholds on validation set ─────────────────────────────────────
    print("\nTuning per-class thresholds on validation set ...")
    val_metrics, val_labels, val_probs = evaluate(model, val_loader)
    if val_metrics['sanitized_batches']:
        print(f"  Validation sanitization count: {val_metrics['sanitized_batches']}")
    thresholds = tune_thresholds(val_labels, val_probs)
    with open(THRESH_PATH, "w") as f:
        json.dump({cls: float(t) for cls, t in zip(ALL_CLASSES, thresholds)}, f, indent=2)
    print(f"  Thresholds saved → {THRESH_PATH}")

    # ── Test set evaluation ───────────────────────────────────────────────────
    print("\nEvaluating on test set ...")
    test_eval_metrics, test_labels, test_probs = evaluate(model, test_loader)
    if test_eval_metrics['sanitized_batches']:
        print(f"  Test sanitization count: {test_eval_metrics['sanitized_batches']}")
    metrics_def  = compute_multilabel_metrics(test_labels, test_probs)
    metrics_tuned = compute_multilabel_metrics(test_labels, test_probs, thresholds)

    print(f"\n  {'Metric':<25} {'Default (0.5)':>14}  {'Tuned thresh':>13}")
    print(f"  {'─'*56}")
    for key in ["mAP", "macro_f1", "micro_f1", "hamming_loss", "exact_match"]:
        d = metrics_def.get(key,   0)
        t = metrics_tuned.get(key, 0)
        print(f"  {key:<25} {d:>14.4f}  {t:>13.4f}")

    print(f"\n  Per-class AP (test, tuned thresholds):")
    print(f"  {'Class':<22} {'AP':>8}  {'Bar'}")
    print(f"  {'─'*50}")
    for cls, ap in metrics_tuned["per_class_ap"].items():
        bar   = "█" * int(ap * 20)
        color = "✓" if ap >= 0.85 else "~" if ap >= 0.70 else "✗"
        print(f"  {color} {cls:<20} {ap:>6.4f}  {bar}")

    # ── Single-label classification report ────────────────────────────────────
    is_single    = test_labels.sum(axis=1) == 1
    sl_labels_gt = test_labels[is_single].argmax(axis=1)
    sl_probs     = test_probs[is_single]
    sl_preds     = sl_probs.argmax(axis=1)
    sl_acc       = float((sl_labels_gt == sl_preds).mean()) if len(sl_labels_gt) else 0.0

    print(f"\n  Single-label test accuracy: {sl_acc:.4f} ({sl_acc*100:.2f}%)")
    report = classification_report(
        sl_labels_gt, sl_preds,
        labels=list(range(NUM_CLASSES)), target_names=ALL_CLASSES,
        digits=4, zero_division=0
    )
    print(f"\n{report}")

    report_path = f"{MODEL_DIR}/classification_report.txt"
    with open(report_path, "w") as f:
        f.write(f"Single-label accuracy: {sl_acc:.4f}\n\n")
        f.write(f"Multi-label metrics (tuned thresholds):\n")
        for k, v in metrics_tuned.items():
            if k != "per_class_ap":
                f.write(f"  {k}: {v:.4f}\n")
        f.write(f"\nClassification Report:\n{report}")
    print(f"  Report saved → {report_path}")

    # ── Confusion matrix (single-label images only) ───────────────────────────
    _plot_confusion_matrix(sl_labels_gt, sl_preds)
    _plot_training_curves()
    _plot_per_class_ap(metrics_tuned["per_class_ap"])

    return metrics_tuned


def _plot_confusion_matrix(true_labels, pred_labels):
    cm = confusion_matrix(true_labels, pred_labels, labels=list(range(NUM_CLASSES)))
    row_sums = cm.sum(axis=1, keepdims=True)
    cm_norm = np.divide(cm.astype(float), row_sums,
                        out=np.zeros_like(cm, dtype=float), where=row_sums != 0)
    fig, ax = plt.subplots(figsize=(15, 13))
    sns.heatmap(cm_norm, annot=cm, fmt="d", cmap="Blues",
                xticklabels=ALL_CLASSES, yticklabels=ALL_CLASSES,
                linewidths=0.5, ax=ax,
                cbar_kws={"label": "Recall (row-normalised)"})
    ax.set_xlabel("Predicted", fontsize=12)
    ax.set_ylabel("True",      fontsize=12)
    ax.set_title("Confusion Matrix — Stage 1 Test Set", fontsize=14)
    plt.xticks(rotation=45, ha="right", fontsize=9)
    plt.yticks(rotation=0,  fontsize=9)
    plt.tight_layout()
    path = f"{PLOTS_DIR}/confusion_matrix.png"
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Confusion matrix → {path}")


def _plot_training_curves():
    if not Path(LOG_PATH).exists(): return
    log     = json.load(open(LOG_PATH))
    epochs  = [e["epoch"]      for e in log]
    tr_loss = [e["train_loss"] for e in log]
    tr_acc  = [e["train_acc"]  for e in log]
    va_map  = [e["val_mAP"]    for e in log]
    va_sl   = [e["val_sl_acc"] for e in log]
    va_f1   = [e["val_macro_f1"] for e in log]

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    axes[0].plot(epochs, tr_loss, marker="o", markersize=3, label="Train loss")
    axes[0].set_title("Training loss"); axes[0].set_xlabel("Epoch")
    axes[0].legend(); axes[0].grid(alpha=0.3)

    axes[1].plot(epochs, va_map, marker="o", markersize=3, label="Val mAP", color="#1D9E75")
    axes[1].plot(epochs, va_sl,  marker="s", markersize=3, label="Val SL acc", color="#378ADD")
    axes[1].set_title("mAP + SL accuracy"); axes[1].set_xlabel("Epoch")
    axes[1].legend(); axes[1].grid(alpha=0.3)

    axes[2].plot(epochs, va_f1, marker="o", markersize=3, label="Val macro F1", color="#7F77DD")
    axes[2].set_title("Macro F1"); axes[2].set_xlabel("Epoch")
    axes[2].legend(); axes[2].grid(alpha=0.3)

    plt.suptitle("EfficientNet-B3 Training — Stage 1", fontsize=13)
    plt.tight_layout()
    path = f"{PLOTS_DIR}/training_curves.png"
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Training curves → {path}")


def _plot_per_class_ap(per_class_ap: dict):
    classes = list(per_class_ap.keys())
    values  = list(per_class_ap.values())
    colors  = ["#1D9E75" if v >= 0.85 else "#EF9F27" if v >= 0.70 else "#E24B4A"
               for v in values]
    fig, ax = plt.subplots(figsize=(14, 6))
    bars = ax.bar(classes, values, color=colors)
    ax.axhline(np.mean(values), color="#378ADD", linestyle="--",
               linewidth=1.5, label=f"mAP={np.mean(values):.3f}")
    ax.set_ylim(0, 1.05); ax.set_ylabel("Average Precision")
    ax.set_title("Per-Class AP — Test Set (green≥0.85, amber≥0.70, red<0.70)")
    ax.legend(); plt.xticks(rotation=45, ha="right")
    for bar, v in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f"{v:.3f}", ha="center", va="bottom", fontsize=9)
    plt.tight_layout()
    path = f"{PLOTS_DIR}/per_class_ap.png"
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Per-class AP  → {path}")


# ══════════════════════════════════════════════════════════════════════════════
# INFERENCE FUNCTION (for deployment)
# ══════════════════════════════════════════════════════════════════════════════

def load_inference_model():
    """
    Load model + thresholds for deployment.
    Returns a callable that takes a PIL image and returns predictions.
    """
    ckpt   = torch.load(MODEL_PATH, map_location=DEVICE)
    model  = build_model()
    model.load_state_dict(ckpt["model_state"])
    model.eval()

    thresholds = np.full(NUM_CLASSES, DEFAULT_THRESH)
    if Path(THRESH_PATH).exists():
        t_dict = json.load(open(THRESH_PATH))
        thresholds = np.array([t_dict.get(cls, DEFAULT_THRESH) for cls in ALL_CLASSES])

    transform = get_transforms("val")

    def predict(pil_image: Image.Image) -> dict:
        img    = transform(pil_image.convert("RGB")).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            logits = torch.nan_to_num(model(img), nan=0.0, posinf=20.0, neginf=-20.0)
            probs  = torch.sigmoid(logits).cpu().numpy()[0]
            probs  = np.nan_to_num(probs, nan=0.0, posinf=1.0, neginf=0.0)
            probs  = np.clip(probs, 0.0, 1.0)

        detected = []
        for cls, prob, thresh in zip(ALL_CLASSES, probs, thresholds):
            if prob >= thresh:
                detected.append({"disease": cls, "confidence": round(float(prob), 4)})

        detected.sort(key=lambda x: -x["confidence"])

        # Optional confidence summary only
        disease_only = [d for d in detected if d["disease"] != "Healthy"]
        if not disease_only:
            confidence_level = "None"
        else:
            mc = max(d["confidence"] for d in disease_only)
            confidence_level = "Low" if mc < 0.65 else "Medium" if mc < 0.85 else "High"

        # Remove Healthy if any non-healthy class is present
        non_healthy = [d for d in detected if d["disease"] != "Healthy"]
        if non_healthy:
            detected = non_healthy

        is_healthy = len(detected) == 1 and detected[0]["disease"] == "Healthy"
        multi_disease = len([d for d in detected if d["disease"] != "Healthy"]) > 1

        return {
            "detected"     : detected,
            "all_probs"    : {cls: round(float(p), 4) for cls, p in zip(ALL_CLASSES, probs)},
            "model_confidence": confidence_level,
            "is_healthy"   : is_healthy,
            "multi_disease": len([d for d in detected if d["disease"] != "Healthy"]) > 1
        }

    return predict


# ══════════════════════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    import ssl
    ssl._create_default_https_context = ssl._create_unverified_context

    train()
    final_evaluate()

✓ Using dataset-specific normalisation stats
CSV_PATH: dataset_balanced/multilabel.csv
DATASET_DIR: dataset_balanced

  Device : mps
  Model  : EfficientNet-B3  |  Input: 300×300
  Epochs : 20  |  Batch: 24  |  LR_head: 0.0003

Loading datasets from multilabel.csv ...
  train:   8340 images  (299 multi-label, 8041 single-label)
  val  :   1275 images  (56 multi-label, 1219 single-label)

  Params total=11,489,845  trainable=793,613 (head only — warmup)

════════════════════════════════════════════════════════════════════════
  PHASE 1 — Warmup (5 epochs, head only, backbone frozen)
════════════════════════════════════════════════════════════════════════
   Ep      Loss    SL_Acc    Val_mAP    Val_SL    Val_F1
    1    1.6649    0.4459     0.6097    0.5414    0.2474  [234s]
    2    1.2740    0.6221     0.6489    0.6411    0.3073  [232s]
    3    1.0173    0.6782     0.7702    0.6919    0.4114  [229s]
    4    0.9262    0.7165     0.7984    0.7299    0.4920  [236s]
    5    0.8974    0.